In [1]:
import numpy as np
import collections
#from gymnasium.envs.registration import register
import gymnasium as gym
import dm_control.suite.swimmer as swimmer
from dm_control.rl import control
from dm_control.utils import rewards
import gym
from gym import spaces
import time


In [2]:
# Shows base tasks from the swimmer environment
print(swimmer.SUITE)

TaggedTasks(OrderedDict([('swimmer6', <function swimmer6 at 0x10fac3e20>), ('swimmer15', <function swimmer15 at 0x10fac3eb0>)]))


In [3]:
# create a custom task that overrides the default swim task to swim at a desired speed 
_SWIM_SPEED = 0.1
class Swim(swimmer.Swimmer):
  """Task to swim forwards at the desired speed."""
  def __init__(self, desired_speed=_SWIM_SPEED, **kwargs):
    super().__init__(**kwargs)
    self._desired_speed = desired_speed
    self._previous_position = None
    self._total_distance_traveled = 0.0

  def initialize_episode(self, physics):
    super().initialize_episode(physics)
    # Hide target by setting alpha to 0.
    physics.named.model.mat_rgba['target', 'a'] = 0
    physics.named.model.mat_rgba['target_default', 'a'] = 0
    physics.named.model.mat_rgba['target_highlight', 'a'] = 0
    self._previous_position = physics.data.qpos.copy()

  def get_observation(self, physics):
    """Returns an observation of joint angles and body velocities."""
    obs = collections.OrderedDict()
    obs['joints'] = physics.joints()
    obs['body_velocities'] = physics.body_velocities()
    return obs

  def get_reward(self, physics):
    """Returns a smooth reward that is 0 when stopped or moving backwards, and rises linearly to 1
    when moving forwards at the desired speed."""
    forward_velocity = -physics.named.data.sensordata['head_vel'][1]
    return rewards.tolerance(
      forward_velocity,
      bounds=(self._desired_speed, float('inf')),
      margin=self._desired_speed,
      value_at_margin=0.,
      sigmoid='linear',
    )

In [4]:
# Register the custom task
@swimmer.SUITE.add()
def free_swim_reece(
  n_links=6,
  desired_speed=_SWIM_SPEED,
  time_limit=swimmer._DEFAULT_TIME_LIMIT,
  random=None,
  environment_kwargs={},
):
  """Returns the Swim task for a n-link swimmer."""
  model_string, assets = swimmer.get_model_and_assets(n_links)

  physics = swimmer.Physics.from_xml_string(model_string, assets=assets)
  task = Swim(desired_speed=desired_speed, random=random)
  return control.Environment(
    physics,
    task,
    time_limit=time_limit,
    control_timestep=swimmer._CONTROL_TIMESTEP,
    **environment_kwargs,
  )

In [5]:
# check that custom tasks is registered 
print(swimmer.SUITE)

TaggedTasks(OrderedDict([('swimmer6', <function swimmer6 at 0x10fac3e20>), ('swimmer15', <function swimmer15 at 0x10fac3eb0>), ('free_swim_reece', <function free_swim_reece at 0x11ea34310>)]))


In [6]:
import numpy as np

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.evaluation import evaluate_policy
from agent.dmc_wrapper import DmControlToGymWrapper

task = swimmer.SUITE['free_swim_reece']
env = task()
gym_env = DmControlToGymWrapper(env)
model = RecurrentPPO("MlpLstmPolicy", gym_env, verbose=1)
model.learn(5000)

vec_env = model.get_env()
mean_reward, std_reward = evaluate_policy(model, vec_env, n_eval_episodes=20, warn=False)
print(mean_reward)

#model.save("ppo_recurrent")

ModuleNotFoundError: No module named 'agent'

In [ ]:
del model # remove to demonstrate saving and loading
model = RecurrentPPO.load("ppo_recurrent")

obs = gym_env.reset()
# cell and hidden state of the LSTM
lstm_states = None
num_envs = 1
# Episode start signals are used to reset the lstm states
episode_starts = np.ones((num_envs,), dtype=bool)
while True:
    action, lstm_states = model.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
    obs, rewards, dones, info = gym_env.step(action)
    episode_starts = dones
    vec_env.render("human")

/Users/reecekeller/miniforge3/envs/mujoco_env/lib/python3.10/site-packages/stable_baselines3/common/vec_env/base_vec_env.py:225: UserWarning: You tried to render a VecEnv with mode='human' but the render mode defined when initializing the environment must be 'human' or 'rgb_array', not 'None'.
  warnings.warn(


AttributeError: 'float' object has no attribute 'tolerance'

: 

In [ ]:

# Optionally, wrap it in a vectorized environment for Stable-Baselines3
#from stable_baselines3.common.vec_env import DummyVecEnv
#gym_env = DummyVecEnv([lambda: gym_env])

# Train an agent using PPO from Stable-Baselines3
import stable_baselines3 as sb3


model = sb3.PPO("MlpPolicy", gym_env, verbose=1)
model.learn(total_timesteps=100000)

# Save the trained model
model.save("ppo_swimmer_model")

# Optionally, evaluate the agent
obs = gym_env.reset()
for _ in range(1000):
    action, _ = model.predict(obs)
    obs, reward, done, info = gym_env.step(action)
    
    if done:
        obs = gym_env.reset()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/Users/reecekeller/miniforge3/envs/mujoco_env/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | 100      |
| time/              |          |
|    fps             | 4540     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | 118         |
| time/                   |             |
|    fps                  | 3320        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011200776 |
|    clip_fraction        | 0.102       |
|    clip_range           | 0.2         |
|    entropy_loss         | -7.08       |
|    explained_variance   | -0.0925     |
|    learning_rate        | 0.

KeyboardInterrupt: 

In [ ]:
# Load the trained model
from dm_control import viewer
model = sb3.PPO.load("ppo_swimmer_model")

# Define a function to run the policy and visualize using viewer.launch
def visualize_policy(env, model, num_steps=1000):
    def policy_step(time_step):
        obs = np.concatenate([time_step.observation['joints'], time_step.observation['body_velocities']], axis=-1)
        action, _ = model.predict(obs)
        return action

    viewer.launch(env, policy=policy_step)

# Visualize the policy
visualize_policy(env, model)

NameError: name 'PPO' is not defined

: 

In [ ]:
import matplotlib.pyplot as plt
from stable_baselines3.common import results_plotter
from stable_baselines3.common.results_plotter import ts2xy, window_func
from stable_baselines3.common.monitor import load_results

def moving_average(values, window):
    """
    Smooth values by doing a moving average
    :param values: (numpy array)
    :param window: (int)
    :return: (numpy array)
    """
    weights = np.repeat(1.0, window) / window
    return np.convolve(values, weights, 'valid')


def plot_results(log_folder, title='Learning Curve'):
    """
    plot the results

    :param log_folder: (str) the save location of the results to plot
    :param title: (str) the title of the task to plot
    """
    x, y = ts2xy(load_results(log_folder), 'timesteps')
    y = moving_average(y, window=50)
    # Truncate x
    x = x[len(x) - len(y):]

    fig = plt.figure(title)
    plt.plot(x, y)
    plt.xlabel('Number of Timesteps')
    plt.ylabel('Rewards')
    plt.title(title + " Smoothed")
    plt.show()

from typing import Callable, List, Optional, Tuple
EPISODES_WINDOW = 100
def plot_curves(
    xy_list: List[Tuple[np.ndarray, np.ndarray]], x_axis: str, title: str, figsize: Tuple[int, int] = (8, 2)
) -> None:
    """
    plot the curves

    :param xy_list: the x and y coordinates to plot
    :param x_axis: the axis for the x and y output
        (can be X_TIMESTEPS='timesteps', X_EPISODES='episodes' or X_WALLTIME='walltime_hrs')
    :param title: the title of the plot
    :param figsize: Size of the figure (width, height)
    """

    plt.figure(title, figsize=figsize)
    max_x = max(xy[0][-1] for xy in xy_list)
    min_x = 0
    for _, (x, y) in enumerate(xy_list):
        plt.scatter(x, y, s=2)
        # Do not plot the smoothed curve at all if the timeseries is shorter than window size.
        if x.shape[0] >= EPISODES_WINDOW:
            # Compute and plot rolling mean with window of size EPISODE_WINDOW
            x, y_mean = window_func(x, y, EPISODES_WINDOW, np.mean)
            plt.plot(x, y_mean, color="r")
    plt.xlim(min_x, max_x)
    plt.title(title)
    plt.xlabel(x_axis)
    plt.ylabel("Episode Rewards")
    plt.tight_layout()


def plot_results_scatter(
    dirs: List[str], num_timesteps: Optional[int], x_axis: str, task_name: str, figsize: Tuple[int, int] = (8, 2)
) -> None:
    """
    Plot the results using csv files from ``Monitor`` wrapper.

    :param dirs: the save location of the results to plot
    :param num_timesteps: only plot the points below this value
    :param x_axis: the axis for the x and y output
        (can be X_TIMESTEPS='timesteps', X_EPISODES='episodes' or X_WALLTIME='walltime_hrs')
    :param task_name: the title of the task to plot
    :param figsize: Size of the figure (width, height)
    """

    data_frames = []
    for folder in dirs:
        data_frame = load_results(folder)
        if num_timesteps is not None:
            data_frame = data_frame[data_frame.l.cumsum() <= num_timesteps]
        data_frames.append(data_frame)
    xy_list = [ts2xy(data_frame, x_axis) for data_frame in data_frames]
    plot_curves(xy_list, x_axis, task_name, figsize)

In [ ]:
plot_results_scatter(['/Users/reecekeller/Documents/neuroagents/zebrafish_agent/curiosity_modules/'], num_timesteps=4000, x_axis = results_plotter.X_TIMESTEPS, task_name = "Easy Swimmer3 No Vision")

LoadMonitorResultsError: No monitor files of the form *monitor.csv found in /Users/reecekeller/Documents/neuroagents/zebrafish_agent/curiosity_modules/

In [ ]:
env = gym.make('Swimmer-v4')

# Define the DDPG model
model = sb3.PPO("MlpPolicy", env, verbose=1)

# Train the model for 10000 steps
model.learn(total_timesteps=10000)

# Save the trained model
model.save("ddpg_swimmer")

# Evaluate the trained model
obs = env.reset()
for i in range(1000):
    action, _states = model.predict(obs)
    obs, rewards, dones, info = env.step(action)
    env.render()

ValueError: XML Error: Schema violation: unrecognized attribute: 'collision'

Element 'option', line 3
